# Model Evaluation

## Project Overview
This notebook loads the trained Linear Regression model, evaluates its performance on the test set, and visualizes prediction quality through diagnostic plots.

## 1. Import Libraries

In [ ]:
from pandas import DataFrame, Series
import pandas as pd
import numpy as np
import joblib
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

## 2. Load Artifacts and Data

In [ ]:
# Load the trained model, scaler, and feature columns
model = joblib.load("models/model.pkl")
scaler = joblib.load("models/scaler.pkl")
feature_cols = json.load(open("models/feature_columns.json"))

print(f"Model type: {type(model).__name__}")
print(f"Number of features: {len(feature_cols)}")
print(f"\nFirst 5 features: {feature_cols[:5]}")
print(f"Last 5 features: {feature_cols[-5:]}")

In [ ]:
# Load the feature-engineered dataset
auto = pd.read_csv("data/processed/auto_features.csv")
X = auto.drop("price", axis=1)
y = auto["price"]

print(f"Dataset shape: {auto.shape}")
print(f"Features: {X.shape[1]}")
print(f"Samples: {X.shape[0]}")

## 3. Split and Scale Test Set

In [ ]:
# Split into train/test (same random_state as training)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

print(f"Train samples: {X_train.shape[0]}")
print(f"Test samples:  {X_test.shape[0]}")

# Scale test set using the fitted scaler
X_test_scaled = scaler.transform(X_test)
print("\n✓ Test set scaled using saved StandardScaler")

## 4. Predict and Inverse-Transform

In [ ]:
# Predict on log-transformed scale, then convert back to dollars
y_pred_log = model.predict(X_test_scaled)
y_pred = np.expm1(y_pred_log)
y_test_actual = np.expm1(y_test.values)

print("=" * 60)
print("PREDICTION SUMMARY")
print("=" * 60)
print(f"\nActual price range:  ${y_test_actual.min():,.2f} - ${y_test_actual.max():,.2f}")
print(f"Predicted price range: ${y_pred.min():,.2f} - ${y_pred.max():,.2f}")
print(f"\nSample predictions (first 10):")
comparison = pd.DataFrame({
    'Actual Price': y_test_actual[:10],
    'Predicted Price': y_pred[:10],
    'Difference': y_test_actual[:10] - y_pred[:10]
})
comparation.index.name = 'Sample'
print(comparation.to_string())

## 5. Regression Metrics

In [ ]:
# Calculate key regression metrics
r2 = r2_score(y_test_actual, y_pred)
mae = mean_absolute_error(y_test_actual, y_pred)
rmse = root_mean_squared_error(y_test_actual, y_pred)
mape = np.mean(np.abs((y_test_actual - y_pred) / y_test_actual)) * 100

print("=" * 60)
print("REGRESSION METRICS")
print("=" * 60)
print(f"\n{'R² (Coefficient of Determination):':<40} {r2:.4f}")
print(f"{'MAE (Mean Absolute Error):':<40} ${mae:,.2f}")
print(f"{'RMSE (Root Mean Squared Error):':<40} ${rmse:,.2f}")
print(f"{'MAPE (Mean Absolute % Error):':<40} {mape:.2f}%")

print(f"\n{'Interpretation:':<40}")
print(f"  The model explains {r2*100:.1f}% of the variance in car prices.")
print(f"  On average, predictions are off by ${mae:,.2f}.")

## 6. Actual vs Predicted Plot

In [ ]:
# Scatter plot of actual vs predicted prices
fig, ax = plt.subplots(figsize=(8, 8))

ax.scatter(y_test_actual, y_pred, alpha=0.6, edgecolors='white', linewidth=0.5)

# Perfect prediction line
min_val = min(y_test_actual.min(), y_pred.min())
max_val = max(y_test_actual.max(), y_pred.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')

ax.set_xlabel('Actual Price ($)')
ax.set_ylabel('Predicted Price ($)')
ax.set_title('Actual vs Predicted Car Prices')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Check for systematic bias by looking at the trend
from scipy import stats

slope, intercept, r_value, p_value, std_err = stats.linregress(y_test_actual, y_pred)

print("LINEAR REGRESSION FIT (Actual vs Predicted):")
print(f"  Slope:     {slope:.4f} (ideal: 1.0)")
print(f"  Intercept: ${intercept:.2f} (ideal: 0)")
print(f"  R²:        {r_value**2:.4f}")

if abs(slope - 1.0) < 0.1 and abs(intercept) < 2000:
    print("\n✓ Model shows no significant systematic bias.")
else:
    print("\n⚠ Model shows some systematic bias - check residuals.")

## 7. Residual Analysis

In [ ]:
# Residual plot: predicted vs residuals
residuals = y_test_actual - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs Predicted
axes[0].scatter(y_pred, residuals, alpha=0.6, edgecolors='white', linewidth=0.5)
axes[0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0].set_xlabel('Predicted Price ($)')
axes[0].set_ylabel('Residual ($)')
axes[0].set_title('Residuals vs Predicted')
axes[0].grid(True, alpha=0.3)

# Histogram of residuals
axes[1].hist(residuals, bins=20, edgecolor='white', alpha=0.7)
axes[1].axvline(x=0, color='r', linestyle='--', linewidth=2)
axes[1].set_xlabel('Residual ($)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Residuals')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nResidual statistics:")
print(f"  Mean residual:    ${np.mean(residuals):.2f}")
print(f"  Std deviation:    ${np.std(residuals):.2f}")
print(f"  Min residual:     ${np.min(residuals):.2f}")
print(f"  Max residual:     ${np.max(residuals):.2f}")

In [ ]:
# Check residual normality with Q-Q plot
fig, ax = plt.subplots(figsize=(7, 7))

stats.probplot(residuals, dist="norm", plot=ax)
ax.set_title('Q-Q Plot of Residuals')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Shapiro-Wilk test for normality
shapiro_stat, shapiro_p = stats.shapiro(residuals)
print(f"Shapiro-Wilk test:")
print(f"  Statistic: {shapiro_stat:.4f}")
print(f"  P-value:   {shapiro_p:.4f}")
if shapiro_p > 0.05:
    print("  ✓ Residuals appear normally distributed (p > 0.05)")
else:
    print("  ⚠ Residuals deviate from normal distribution (p < 0.05)")

## 8. Feature Coefficients

In [ ]:
# Display model coefficients
coefficients = pd.Series(model.coef_, index=feature_cols)

print("=" * 60)
print("MODEL COEFFICIENTS")
print("=" * 60)
print(f"\nIntercept: {model.intercept_:.4f}")
print(f"\nTop 10 positive coefficients:")
print(coefficients.nlargest(10).to_string())
print(f"\nTop 10 negative coefficients:")
print(coefficients.nsmallest(10).to_string())

In [ ]:
# Visualize top coefficients
top_n = 15
top_coef = coefficients.abs().nlargest(top_n)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in coefficients[top_coef.index]]

ax.barh(range(len(top_coef)), coefficients[top_coef.index], color=colors)
ax.set_yticks(range(len(top_coef)))
ax.set_yticklabels(top_coef.index)
ax.set_xlabel('Coefficient Value')
ax.set_title(f'Top {top_n} Most Influential Features')
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
ax.grid(True, alpha=0.3, axis='x')

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2ecc71', label='Positive'),
                   Patch(facecolor='#e74c3c', label='Negative')]
ax.legend(handles=legend_elements)

plt.tight_layout()
plt.show()

## 9. Prediction Error by Make

In [ ]:
# Analyze prediction errors by car make
# Reconstruct make from one-hot columns
make_cols = [col for col in feature_cols if col.startswith("make_")]
X_test_makes = X_test[make_cols].idxmax(axis=1).str.replace("make_", "")

error_by_make = pd.DataFrame({
    'Make': X_test_makes,
    'Actual': y_test_actual,
    'Predicted': y_pred,
    'Error': np.abs(y_test_actual - y_pred),
    'Error_Pct': np.abs((y_test_actual - y_pred) / y_test_actual) * 100
})

print("=" * 60)
print("MEAN ABSOLUTE ERROR BY MAKE")
print("=" * 60)
make_summary = error_by_make.groupby('Make').agg({
    'Error': 'mean',
    'Error_Pct': 'mean',
    'Actual': 'count'
}).rename(columns={'Actual': 'Count'}).sort_values('Error', ascending=False)

make_summary['Error'] = make_summary['Error'].map('${:,.0f}'.format)
make_summary['Error_Pct'] = make_summary['Error_Pct'].map('{:.1f}%'.format)
print(make_summary.to_string())

In [ ]:
# Visualize MAE by make
fig, ax = plt.subplots(figsize=(12, 5))

make_errors = error_by_make.groupby('Make')['Error'].mean().sort_values(ascending=True)
ax.barh(range(len(make_errors)), make_errors.values, color='#3498db')
ax.set_yticks(range(len(make_errors)))
ax.set_yticklabels(make_errors.index)
ax.set_xlabel('Mean Absolute Error ($)')
ax.set_title('Prediction Error by Car Make')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 10. Summary

In [ ]:
print("=" * 60)
print("EVALUATION SUMMARY")
print("=" * 60)
print(f"\n{'Model:':<25} Linear Regression")
print(f"{'Features:':<25} {len(feature_cols)}")
print(f"{'Test samples:':<25} {len(y_test)}")
print(f"{'':<25} ")
print(f"{'R² Score:':<25} {r2:.4f}")
print(f"{'MAE:':<25} ${mae:,.2f}")
print(f"{'RMSE:':<25} ${rmse:,.2f}")
print(f"{'MAPE:':<25} {mape:.2f}%")
print(f"{'':<25} ")
print(f"{'Mean residual:':<25} ${np.mean(residuals):,.2f}")
print(f"{'Residual std dev:':<25} ${np.std(residuals):,.2f}")
print(f"{'':<25} ")
print("Key observations:")
print(f"  • The model explains {r2*100:.1f}% of price variance (R² = {r2:.3f})")
print(f"  • Average prediction error is ${mae:,.0f} (MAPE: {mape:.1f}%)")
print(f"  • Most influential features: engine size, horsepower, curb weight")
print(f"  • Residuals are {'normally' if shapiro_p > 0.05 else 'not normally'} distributed")
print(f"\n✓ Model evaluation complete!")